# Replace an expensive LLM with a small model you own: end-to-end on a free T4

Runs every stage of [https://github.com/isarinio22/llms](https://github.com/isarinio22/llms) on Colab. Runtime: **T4 GPU**.

* The repo is cloned **into Google Drive**, so data splits, the teacher cache, checkpoints, models and results
  survive a disconnect. After a disconnect, run the first two cells again, then continue: finished stages are skipped.
* Packages go into an isolated venv (`/content/venv`) with the exact pins from `requirements-gpu.lock`,
  so they never clash with Colab's preinstalled packages and no runtime restart is needed.
* Run Colab under the Google account whose Drive you want to use: Drive can only be mounted for the account
  that runs the notebook (top-right avatar).
* Each stage runs in its own process, so GPU memory is fully released between stages.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!test -d /content/drive/MyDrive/llms/.git || git clone https://github.com/isarinio22/llms /content/drive/MyDrive/llms
%cd /content/drive/MyDrive/llms
!git pull --ff-only

In [ ]:
%%bash
# Isolated environment with every package pinned (requirements-gpu.lock). A few minutes per fresh session.
# Stops at the first error; the last line must print the versions, otherwise do not continue.
set -euo pipefail
pip install -q uv
test -x /content/venv/bin/python || uv venv /content/venv --python 3.12 -q
uv pip install --python /content/venv/bin/python -r requirements-gpu.lock
uv pip install -q --python /content/venv/bin/python -e . --no-deps
/content/venv/bin/python -c "import yaml, torch, vllm, llms; print('OK torch', torch.__version__, '| vllm', vllm.__version__, '|', torch.cuda.get_device_name(0))"

In [ ]:
CONFIG = "configs/base.yaml"   # or an ablation, e.g. configs/ablations/student_1p7b.yaml
EXTRA = ""                       # e.g. "--set teacher.backend=hf" if the vLLM probe fails

## `doctor`

Environment report and vLLM probe on the T4. If a probe fails, add `--set teacher.backend=hf` to the teacher stage.

In [ ]:
!/content/venv/bin/python -m llms run --stage doctor --config $CONFIG $EXTRA

## `data`

Stratified train-pool / unlabeled / dev / test split before any teacher call, then cross-split near-duplicate removal.

In [ ]:
!/content/venv/bin/python -m llms run --stage data --config $CONFIG $EXTRA

## `teacher_label`

Teacher labels with k=3 self-consistency. Cached after every chunk: rerun this cell after a disconnect and it resumes.

In [ ]:
!/content/venv/bin/python -m llms run --stage teacher_label --config $CONFIG $EXTRA

## `baseline`

Embeddings + logistic regression, trained on teacher labels and (as a reference) on gold labels.

In [ ]:
!/content/venv/bin/python -m llms run --stage baseline --config $CONFIG $EXTRA

## `train_student`

LoRA SFT on filtered teacher labels, early stopping on dev macro-F1, merged model for serving.

In [ ]:
!/content/venv/bin/python -m llms run --stage train_student --config $CONFIG $EXTRA

## `learning_curve`

The same recipe on 500 / 1,000 / 2,000 examples.

In [ ]:
!/content/venv/bin/python -m llms run --stage learning_curve --config $CONFIG $EXTRA

## `score`

Score all 77 labels for dev / test / unlabeled; temperature scaling on dev.

In [ ]:
!/content/venv/bin/python -m llms run --stage score --config $CONFIG $EXTRA

## `evaluate`

Final test evaluation and paired statistics (McNemar, non-inferiority, sample-size planner).

In [ ]:
!/content/venv/bin/python -m llms run --stage evaluate --config $CONFIG $EXTRA

## `cascade`

Confidence cascade and cost model.

In [ ]:
!/content/venv/bin/python -m llms run --stage cascade --config $CONFIG $EXTRA

## `quantize`

bitsandbytes 8-bit and 4-bit: paired non-inferiority against the fp16 student.

In [ ]:
!/content/venv/bin/python -m llms run --stage quantize --config $CONFIG $EXTRA

## `serve_bench`

vLLM server vs Hugging Face generate: throughput, latency, time to first token.

In [ ]:
!/content/venv/bin/python -m llms run --stage serve_bench --config $CONFIG $EXTRA

## `report`

Render every table into README.md from results/full/*.json.

In [ ]:
!/content/venv/bin/python -m llms run --stage report --config $CONFIG $EXTRA

## Figures

In [ ]:
from IPython.display import Image, Markdown, display
import glob
for f in sorted(glob.glob("figures/full/*.png")):
    display(Markdown(f"**{f}**")); display(Image(f))

## Save the results to GitHub

`results/full/` and `figures/full/` are small and meant to be committed; `artifacts/` is gitignored.
Commit from here (needs a token) or download the two folders and commit locally.

In [ ]:
!zip -qr /content/results_full.zip results/full figures/full README.md
from google.colab import files
files.download("/content/results_full.zip")